# Symmetry of the Y₂Ir₂O₇ models (paper `tab:symmetry-yio`, Fig. 6b)

Do the Wannier models behind ∂θ/∂Q obey the magnetic point group of their structure? Production window
`f12.422_w22.644_k8`: base 220455 and the Q = 0.01 Å Γ₂⁻ patterns, with the models of the production c₂ jobs
(`axion_phonon.wannier.pair_models`: H and A(R) from the `.chk`, pinned to the base atoms, one τ for both).

The tests run on the cluster (the displaced structures' `.chk` are only there): `tests/symmetry/run_yio_symmetry.{sh,py}`,
one job per pattern (2026-11-03, jobs 220828–220830), each testing the base and its pattern:
1. DFT eigenvalues of the lowest 260 bands on the 8³ mesh: e_n(g k) = e_n(k), k → −g k for a primed g;
2. interpolated bands at 7 random k;
3. the band-traced Berry curvature ω_a = Re Tr F_a of the 156 occupied bands (AIAO breaks time reversal, so ω ≠ 0):
   ω(g k) = det(R) R ω(k), with −det(R) for a primed g; separately for the internal, cross and external parts and their sum.

The groups come from each structure's pw.x output: base m-3m′ (48 operations), patterns −4′3m′ (24). Controls: C4z and
S4z belong to neither group (must fail, ~200%); C4z·T belongs to the base group and is broken by the displacement
(must exceed 10× the largest residual). This notebook reads the jobs' `symmetry_{base,mode}.npz` (copied to `data/`)
and prints the table. The local check of the base alone is `tests/symmetry/test_yio_symmetry.py`.


In [1]:
# ---- settings
from pathlib import Path
WINDOW = Path("/Users/treycole/Repos/axion-phonon/data/Y2Ir2O7/windows/f12.422_w22.644_k8")
PATTERNS = ("GM2-_p1_Q0.01", "GM2-_p2_Q0.01", "GM2-_p3_Q0.01")
RESULTS = {s: WINDOW / f"symmetry_{s}/out" for s in PATTERNS}   # symmetry_base.npz, symmetry_mode.npz, symmetry.log
# ---------------


In [2]:
import numpy as np

PARTS = ("internal", "cross", "external", "total")


def summary(f):
    r = dict(np.load(f, allow_pickle=True))
    n = int(r["n_ops"]); rows = r["rows"]; sym = rows[1:n]
    ctl = {str(nm).replace("control ", ""): (v, bool(b)) for nm, v, b in
           zip(r["names"][n:], rows[n:, 4], r.get("control_broken", np.zeros(len(rows) - n, bool)))}
    return dict(n=n, dft=r["dft_ueV"][1:].max(), bands_meV=sym[:, 0].max(), med=np.median(sym[:, 1:], 0),
                mx=sym[:, 1:].max(0), ctl=ctl)


def row(label, s):
    return (f"{label:22s} {s['n']:3d} ops  DFT {s['dft']:.2f} ueV  bands {s['bands_meV'] * 1e3:.2f} ueV  "
            + "  ".join(f"{p} {100 * a:.4f}% ({100 * b:.4f}%)" for p, a, b in zip(PARTS, s["med"], s["mx"]))
            + "  | " + ", ".join(f"{k} {100 * v:.1f}%{' (broken)' if b else ''}" for k, (v, b) in s["ctl"].items()))


for i, (s, folder) in enumerate(RESULTS.items()):
    if not (folder / "symmetry_mode.npz").exists():
        print(f"{s}: no results in {folder}"); continue
    if i == 0:
        print(row("undistorted", summary(folder / "symmetry_base.npz")))
    print(row(s, summary(folder / "symmetry_mode.npz")))


undistorted             48 ops  DFT 1.35 ueV  bands 0.60 ueV  internal 0.0002% (0.0003%)  cross 0.0069% (0.0097%)  external 0.0060% (0.0094%)  total 0.0003% (0.0004%)  | C4z 200.0%, S4z 200.0%
GM2-_p1_Q0.01           24 ops  DFT 1.34 ueV  bands 0.56 ueV  internal 0.0002% (0.0003%)  cross 0.0070% (0.0107%)  external 0.0067% (0.0097%)  total 0.0003% (0.0005%)  | C4z 200.0%, C4z T 1.8% (broken), S4z 199.4%
GM2-_p2_Q0.01           24 ops  DFT 1.31 ueV  bands 0.71 ueV  internal 0.0002% (0.0003%)  cross 0.0068% (0.0099%)  external 0.0064% (0.0101%)  total 0.0003% (0.0004%)  | C4z 200.0%, C4z T 11.6% (broken), S4z 196.4%
GM2-_p3_Q0.01           24 ops  DFT 1.28 ueV  bands 0.57 ueV  internal 0.0002% (0.0003%)  cross 0.0066% (0.0079%)  external 0.0067% (0.0099%)  total 0.0003% (0.0005%)  | C4z 200.0%, C4z T 3.0% (broken), S4z 199.0%
